# 07 — Single-Cell Fragment Matrices & Pseudobulk Aggregation

Build **single-cell count matrices** (peaks × barcodes) per species from raw
fragment files, using the **cross-species consensus peak set** (liftback
coordinates).  Then aggregate into **pseudobulk** count vectors grouped by
cell type, donor, and experimental run for downstream DESeq2 analysis.

## Workflow
1. Load metadata (`{Species}_RNA.tsv`) and QC-passing barcodes
2. For each species × sample: intersect fragments with consensus peaks →
   sparse count matrix
3. Concatenate per-sample matrices into a single species-level matrix
4. Create pseudobulk by summing counts over grouping variables
5. Save single-cell and pseudobulk matrices to disk (feather / parquet)

> **Note:** This notebook replaces the per-species `91_{Species}_filtered_bigwigs`
> notebooks with a single, unified workflow.

In [ ]:
# === IMPORTS ===
import os
import re
import sys
import gzip
import glob
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import scipy.sparse as sp_sparse

warnings.simplefilter(action="ignore", category=FutureWarning)

# pycisTopic fragments / matrix utilities
import polars as pl
pl.enable_string_cache()

import pyarrow as pa
import pyarrow.csv

try:
    import pyranges as pr
except ImportError:
    pr = None

from pycisTopic.fragments import (
    read_fragments_to_polars_df,
    read_bed_to_polars_df,
    filter_fragments_by_cb,
)
from pycisTopic.genomic_ranges import intersection as gr_intersection

# Pipeline utilities
sys.path.insert(0, os.path.abspath(".."))

print("✅  Imports OK")

## 1 — Configuration

All paths (genomes, fragments, metadata, consensus peaks, output) are
defined centrally here.  Modify this cell once to match your setup.

In [ ]:
# === PATHS ===
BASE        = "/cluster/project/treutlein/USERS/jjans"
WORK        = "/cluster/work/treutlein/jjans"
GENOMES_REF = f"{WORK}/data/intestine/nhp_atlas/genomes/reference_"

# Cross-species consensus peak set (from 04_cross_species_consensus)
CROSS_SPECIES_DIR = f"{BASE}/analysis/adult_intestine/peaks/cross_species_consensus_v2"
UNIFIED_PEAKS     = os.path.join(CROSS_SPECIES_DIR, "02_merged_consensus",
                                 "unified_consensus_hg38_with_ids.bed")
LIFTBACK_DIR      = os.path.join(CROSS_SPECIES_DIR, "04_lifted_back")

# Metadata
META_DIR  = f"{BASE}/analysis/adult_intestine/rna/integration/meta_data"
QC_DIR    = f"{BASE}/analysis/adult_intestine/atac"

# Raw fragment files
NHP_FRAG_DIR   = f"{WORK}/data/intestine/nhp_atlas/atac"
HUMAN_FRAG_DIR = f"{WORK}/data/intestine/human_ref_atlas/atac"

# Output
OUTPUT_DIR = os.path.join(CROSS_SPECIES_DIR, "12_fragment_matrices")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Species definitions ---
SPECIES_LIST = ["Bonobo", "Chimpanzee", "Gorilla", "Macaque", "Marmoset", "Human"]

# Chromosome sizes (same as 05_quantification)
CHROM_SIZES = {
    "Bonobo":      f"{GENOMES_REF}/panPan1/fasta/panPan1.chrom.sizes",
    "Chimpanzee":  f"{GENOMES_REF}/panTro3/fasta/panTro3.chrom.sizes",
    "Gorilla":     f"{GENOMES_REF}/gorGor4/fasta/gorGor4.chrom.sizes",
    "Macaque":     f"{GENOMES_REF}/Mmul10/fasta/Mmul10.chrom.sizes",
    "Marmoset":    f"{GENOMES_REF}/calJac1_mito/fasta/calJac1_mito.chrom.sizes",
    "Human":       f"{BASE}/analysis/intestine/encode_data/hg38.chrom.sizes",
}

# Per-species liftback consensus peak files
SPECIES_PEAK_FILES = {}
for _sp in SPECIES_LIST:
    if _sp == "Human":
        _pf = UNIFIED_PEAKS
    else:
        _pf = os.path.join(LIFTBACK_DIR, f"unified_consensus_{_sp}.bed")
    if os.path.exists(_pf):
        SPECIES_PEAK_FILES[_sp] = _pf

# NHP sample → species mapping (from original notebooks)
NHP_SAMPLES = {
    "Bonobo":      ["Sample_086", "Sample_093", "Sample_094", "Sample_121",
                    "Sample_122", "Sample_127", "Sample_128", "Sample_139",
                    "Sample_140", "Sample_175"],
    "Chimpanzee":  ["Sample_087", "Sample_088", "Sample_095", "Sample_096",
                    "Sample_119", "Sample_120", "Sample_125", "Sample_126",
                    "Sample_174"],
    "Gorilla":     ["Sample_081", "Sample_082", "Sample_117", "Sample_118",
                    "Sample_123", "Sample_124", "Sample_168", "Sample_169",
                    "Sample_170", "Sample_171", "Sample_172", "Sample_173"],
    "Macaque":     ["Sample_131", "Sample_132", "Sample_133", "Sample_134",
                    "Sample_137", "Sample_138"],
    "Marmoset":    ["Sample_089", "Sample_090", "Sample_091", "Sample_092",
                    "Sample_097", "Sample_098", "Sample_099", "Sample_100",
                    "Sample_129", "Sample_130", "Sample_135", "Sample_136"],
}

print(f"Output directory: {OUTPUT_DIR}")
print(f"Unified peak set: {UNIFIED_PEAKS}")
for sp, pf in SPECIES_PEAK_FILES.items():
    n = sum(1 for _ in open(pf))
    print(f"  {sp:<15} {n:>9,} peaks   {os.path.basename(pf)}")

## 2 — Build fragment and metadata dictionaries per species

For each species we need:
- `fragments_dict`: sample_id → path to `.fragments.tsv.gz`
- `cell_data`: barcode-indexed metadata DataFrame, filtered to Adult + QC-passing barcodes

In [ ]:
def _reindex_nhp(cell_data: pd.DataFrame) -> pd.DataFrame:
    """Re-index NHP metadata so barcodes match ``{barcode}-{sample}`` format.

    Original index: ``Sample_086_AAACAGCCAAGGAATC-1``
    Re-indexed:     ``AAACAGCCAAGGAATC-1-Sample_086``
    """
    cell_data = cell_data.copy()
    cell_data["old_barcode"] = cell_data.index
    idx = cell_data.index.to_series()
    idx = idx.str.replace("_plus_resequenced", "", regex=False)
    idx = idx.str.replace("Sample_", "Sample", regex=False)
    idx = idx.apply(lambda x: f"{x.split('_')[1]}-{x.split('_')[0]}")
    idx = idx.str.replace("Sample", "Sample_", regex=False)
    idx = idx.str.replace("MSN", "Sample_", regex=False)
    cell_data.index = idx
    return cell_data


def _reindex_human(cell_data: pd.DataFrame) -> pd.DataFrame:
    """Re-index Human metadata so barcodes match ``{barcode}-{sample}`` format.

    Original index: ``B006-A-002#AAACAGCCAACTAGCC-1``
    Re-indexed:     ``AAACAGCCAACTAGCC-1-B006_A_002``
    """
    cell_data = cell_data.copy()
    idx = cell_data.index.to_series()
    idx = idx.str.replace("_plus_resequenced", "", regex=False)
    idx = idx.str.replace("Sample_", "Sample", regex=False)
    # Human: "B006-A-002#AAACAGCCAACTAGCC-1" → "AAACAGCCAACTAGCC-1-B006_A_002"
    idx = idx.apply(lambda x: f"{x.split('#')[1]}-{x.split('#')[0].replace('-', '_')}")
    cell_data.index = idx
    # Normalise dashes → underscores in sample-related columns
    for col in ["orig.ident", "Sample.ID", "SampleNameOnly"]:
        if col in cell_data.columns:
            cell_data[col] = cell_data[col].str.replace("-", "_", regex=False)
    return cell_data


def load_species_data(species: str) -> tuple[pd.DataFrame, dict]:
    """
    Return (cell_data, fragments_dict) for one species.

    cell_data is filtered to Adult + QC-passing barcodes.
    Barcodes are in ``{raw_bc}-{sample}`` format, guaranteed unique.
    """
    # --- Load metadata ---
    meta_path = os.path.join(META_DIR, f"{species}_RNA.tsv")
    cell_data = pd.read_csv(meta_path, sep="\t", index_col=0)
    n_total = len(cell_data)

    # Re-index so barcode = {raw_bc}-{sample}
    if species == "Human":
        cell_data = _reindex_human(cell_data)
    else:
        cell_data = _reindex_nhp(cell_data)

    # Filter to Adult
    cell_data = cell_data[cell_data["Age"] == "Adult"].copy()
    n_adult = len(cell_data)

    # --- Build fragments dict: sample → fragment file path ---
    if species == "Human":
        frag_files = glob.glob(os.path.join(HUMAN_FRAG_DIR, "*_atac_fragments.tsv.gz"))
        fragments_dict = {
            os.path.basename(f).split("_atac_fragments")[0].replace("-", "_"): f
            for f in frag_files
        }
        valid_samples = set(cell_data["orig.ident"].unique())
        fragments_dict = {k: v for k, v in fragments_dict.items() if k in valid_samples}
    else:
        fragments_dict = {
            s: os.path.join(NHP_FRAG_DIR, f"{s}_atac.fragments.tsv.gz")
            for s in NHP_SAMPLES[species]
        }

    # --- QC barcode filtering ---
    qc_species_dir = os.path.join(QC_DIR, f"qc_{species}")
    samples = sorted(set(cell_data["orig.ident"]))
    all_qc_barcodes = []
    qc_counts = {}

    for sample in samples:
        qc_file = os.path.join(qc_species_dir, f"{sample}.cbs_for_otsu_thresholds.tsv")
        if not os.path.exists(qc_file):
            print(f"    ⚠️  QC file not found: {qc_file}")
            continue
        qc_df = pd.read_csv(qc_file, sep="\t", header=None)
        barcodes = [f"{bc}-{sample}" for bc in qc_df[0].tolist()]
        qc_counts[sample] = len(barcodes)
        all_qc_barcodes.extend(barcodes)

    # Intersect QC barcodes with metadata barcodes
    keep = sorted(set(all_qc_barcodes) & set(cell_data.index))
    cell_data = cell_data.loc[keep]

    # Verify barcode uniqueness
    n_unique = cell_data.index.nunique()
    n_total_idx = len(cell_data.index)
    if n_unique != n_total_idx:
        n_dupes = n_total_idx - n_unique
        print(f"    ⚠️  {n_dupes} duplicate barcodes detected after QC filter!")
        # Keep first occurrence only
        cell_data = cell_data[~cell_data.index.duplicated(keep="first")]

    # Keep only samples that still have cells
    remaining_samples = sorted(cell_data["orig.ident"].unique())
    fragments_dict = {s: fragments_dict[s] for s in remaining_samples
                      if s in fragments_dict}

    # Diagnostic: show barcode format
    if len(cell_data) > 0:
        ex_bc = cell_data.index[0]
        ex_sample = cell_data.iloc[0]["orig.ident"]
        print(f"  Barcode format: '{ex_bc}' (sample='{ex_sample}')")

    return cell_data, fragments_dict


# === Load all species ===
ALL_CELL_DATA = {}
ALL_FRAG_DICTS = {}

for species in SPECIES_LIST:
    print(f"\n{'='*60}")
    print(f"  {species}")
    print(f"{'='*60}")
    cd, fd = load_species_data(species)
    ALL_CELL_DATA[species] = cd
    ALL_FRAG_DICTS[species] = fd

    print(f"  Cells:    {len(cd):>8,}  (unique: {cd.index.nunique():,})")
    print(f"  Samples:  {len(fd):>8}")
    if "cell_type_initial" in cd.columns:
        print(f"  Types:    {cd['cell_type_initial'].nunique():>8}")
    if "Individual" in cd.columns:
        print(f"  Donors:   {cd['Individual'].nunique():>8}")
    # Show per-sample breakdown
    for s in sorted(fd.keys())[:5]:
        n = (cd["orig.ident"] == s).sum()
        print(f"    {s}: {n:,} cells → {fd[s]}")
    if len(fd) > 5:
        print(f"    ... and {len(fd) - 5} more samples")

## 3 — Create single-cell fragment count matrices

For each species we:
1. Load the liftback consensus peaks as a Polars DataFrame
2. For each sample, read fragments, filter to QC barcodes, intersect with peaks
3. Build a sparse CSR matrix (peaks × barcodes) per sample
4. Concatenate across samples into one species-level matrix

In [ ]:
def load_regions_as_polars(peak_file: str) -> pl.DataFrame:
    """Load a BED file and add RegionID column (as Utf8 for safe joining)."""
    regions = read_bed_to_polars_df(peak_file, engine="pyarrow", min_column_count=3)
    regions = regions.with_columns(
        (
            pl.col("Chromosome").cast(pl.Utf8)
            + ":" + pl.col("Start").cast(pl.Utf8)
            + "-" + pl.col("End").cast(pl.Utf8)
        ).alias("RegionID")
    ).select(["Chromosome", "Start", "End", "RegionID"])
    return regions


def _harmonize_chroms(
    regions_pl: pl.DataFrame,
    fragments_pl: pl.DataFrame,
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """
    Ensure Chromosome columns use the same naming convention.

    Strategy: if one has 'chr' prefix and the other doesn't, we strip
    'chr' from whichever has it.  Returns (regions, fragments, regions_changed).
    """
    reg_chroms = set(regions_pl.get_column("Chromosome").unique().to_list())
    frag_chroms = set(fragments_pl.get_column("Chromosome").unique().to_list())

    # Already compatible?
    if reg_chroms & frag_chroms:
        return regions_pl, fragments_pl, False

    reg_has_chr = any(str(c).startswith("chr") for c in reg_chroms)
    frag_has_chr = any(str(c).startswith("chr") for c in frag_chroms)

    if reg_has_chr and not frag_has_chr:
        # Strip 'chr' from peaks to match fragments
        regions_pl = regions_pl.with_columns(
            pl.col("Chromosome").cast(pl.Utf8).str.replace("^chr", "")
            .cast(pl.Categorical).alias("Chromosome")
        )
        regions_pl = regions_pl.with_columns(
            (
                pl.col("Chromosome").cast(pl.Utf8)
                + ":" + pl.col("Start").cast(pl.Utf8)
                + "-" + pl.col("End").cast(pl.Utf8)
            ).alias("RegionID")
        )
        print("    [chr harmonize] Stripped 'chr' from peaks to match fragments")
        return regions_pl, fragments_pl, True

    elif frag_has_chr and not reg_has_chr:
        # Strip 'chr' from fragments to match peaks
        fragments_pl = fragments_pl.with_columns(
            pl.col("Chromosome").cast(pl.Utf8).str.replace("^chr", "")
            .cast(pl.Categorical).alias("Chromosome")
        )
        print("    [chr harmonize] Stripped 'chr' from fragments to match peaks")
        return regions_pl, fragments_pl, False

    return regions_pl, fragments_pl, False


def build_fragment_matrix(
    species: str,
    cell_data: pd.DataFrame,
    fragments_dict: dict,
    peak_file: str,
) -> tuple:
    """
    Build a sparse (peaks × barcodes) count matrix for one species.

    Barcodes in cell_data.index have the format ``{raw_barcode}-{sample}``
    (e.g. ``AACG…-1-Sample_086``).  The raw barcode part (``AACG…-1``)
    matches the CB column in fragment files.  The sample suffix is stripped
    using the known sample name (not rsplit), so dashes inside sample names
    are handled correctly.

    Returns (csr_matrix, barcode_list, region_id_list, metadata_df).
    """
    # ---- load peak regions -------------------------------------------------
    regions_pl = load_regions_as_polars(peak_file)
    region_ids = regions_pl.get_column("RegionID").to_list()
    n_regions = len(region_ids)
    region_to_idx = {rid: i for i, rid in enumerate(region_ids)}

    # ---- accumulators -------------------------------------------------------
    all_barcodes: list[str] = []
    all_rows: list[int] = []
    all_cols: list[int] = []
    all_vals: list[int] = []
    meta_records: list[dict] = []

    has_individual = "Individual" in cell_data.columns
    has_run = "Sequencing.Run" in cell_data.columns
    has_celltype = "cell_type_initial" in cell_data.columns

    _first_sample = True

    for sample_idx, (sample, frag_path) in enumerate(sorted(fragments_dict.items())):
        if not os.path.exists(frag_path):
            print(f"  ⚠️  Fragment file not found: {frag_path}")
            continue

        # -- barcodes for this sample -----------------------------------------
        sample_suffix = f"-{sample}"
        sample_barcodes = cell_data[cell_data["orig.ident"] == sample].index.tolist()
        if not sample_barcodes:
            continue

        print(f"  [{sample_idx+1}/{len(fragments_dict)}] {sample}: "
              f"{len(sample_barcodes):,} barcodes", end="", flush=True)

        # Extract raw barcodes by stripping the known sample suffix
        # e.g. "AAAC…-1-Sample_086"  →  "AAAC…-1"
        cbs_raw = []
        for bc in sample_barcodes:
            if bc.endswith(sample_suffix):
                cbs_raw.append(bc[: -len(sample_suffix)])
            else:
                cbs_raw.append(bc)  # fallback
        cbs_series = pl.Series("CB", cbs_raw, dtype=pl.Categorical)

        # -- read & filter fragments ------------------------------------------
        fragments_pl = read_fragments_to_polars_df(
            fragments_bed_filename=frag_path,
            engine="pyarrow",
        )
        fragments_filt = filter_fragments_by_cb(fragments_pl, cbs_series)
        del fragments_pl

        if fragments_filt.height == 0:
            print(" → 0 fragments after CB filter")
            continue

        # -- chromosome harmonisation -----------------------------------------
        # Called per-sample; cheap no-op when chroms already match.
        # On first mismatch it modifies regions or fragments once; subsequent
        # calls with the same convention return immediately.
        regions_pl, fragments_filt, regions_changed = _harmonize_chroms(
            regions_pl, fragments_filt
        )
        if regions_changed:
            region_ids = regions_pl.get_column("RegionID").to_list()
            region_to_idx = {rid: i for i, rid in enumerate(region_ids)}

        # -- intersect fragments with peaks -----------------------------------
        overlap_df = gr_intersection(
            regions1_df_pl=regions_pl,
            regions2_df_pl=fragments_filt,
            how="all",
            regions1_info=True,
            regions2_info=True,
            regions1_coord=False,
            regions2_coord=False,
            regions1_suffix="@1",
            regions2_suffix="@2",
        )

        if _first_sample:
            print(f"\n    [DEBUG] intersection columns: {overlap_df.columns}")
            print(f"    [DEBUG] intersection rows:    {overlap_df.height:,}")
            _first_sample = False

        if overlap_df.height == 0:
            print(" → 0 intersections")
            continue

        # -- find column names in intersection output -------------------------
        region_col = next(
            (c for c in ["RegionID@1", "RegionID"] if c in overlap_df.columns),
            None,
        )
        cb_col = next(
            (c for c in ["Name@2", "CB@2", "Name", "CB"] if c in overlap_df.columns),
            None,
        )
        if region_col is None or cb_col is None:
            print(f" → ⚠️ missing columns (RegionID={region_col}, CB={cb_col})")
            print(f"       available: {overlap_df.columns}")
            continue

        # -- count overlaps per region × barcode ------------------------------
        region_cb_counts = (
            overlap_df.lazy()
            .group_by([region_col, cb_col])
            .agg(pl.len().cast(pl.UInt32).alias("count"))
            .collect()
        )
        del overlap_df, fragments_filt

        # -- barcode → global column index ------------------------------------
        unique_cbs_raw = sorted(
            region_cb_counts.get_column(cb_col).unique().to_list()
        )
        local_bc_map: dict[str, int] = {}
        sample_barcodes_set = set(sample_barcodes)

        for raw_bc in unique_cbs_raw:
            full_bc = f"{raw_bc}{sample_suffix}"
            if full_bc in sample_barcodes_set:
                local_bc_map[raw_bc] = len(all_barcodes)
                all_barcodes.append(full_bc)

                rec = {"barcode": full_bc, "sample": sample}
                if full_bc in cell_data.index:
                    if has_celltype:
                        rec["cell_type"] = cell_data.at[full_bc, "cell_type_initial"]
                    if has_individual:
                        rec["donor"] = cell_data.at[full_bc, "Individual"]
                    if has_run:
                        rec["run"] = cell_data.at[full_bc, "Sequencing.Run"]
                meta_records.append(rec)

        # -- sparse triplets --------------------------------------------------
        region_id_vals = region_cb_counts.get_column(region_col).to_list()
        cbs_vals = region_cb_counts.get_column(cb_col).to_list()
        count_vals = region_cb_counts.get_column("count").to_numpy()

        for i in range(len(region_id_vals)):
            bc = cbs_vals[i]
            rid = region_id_vals[i]
            if bc in local_bc_map and rid in region_to_idx:
                all_rows.append(region_to_idx[rid])
                all_cols.append(local_bc_map[bc])
                all_vals.append(int(count_vals[i]))

        print(f" → {len(local_bc_map):,} barcodes with signal")

    # ---- assemble sparse matrix ---------------------------------------------
    n_barcodes = len(all_barcodes)
    if n_barcodes == 0:
        print("  ⚠️  No barcodes found!")
        return None, [], region_ids, pd.DataFrame()

    # Verify barcode uniqueness
    if len(set(all_barcodes)) != n_barcodes:
        from collections import Counter
        dupes = [bc for bc, cnt in Counter(all_barcodes).items() if cnt > 1]
        print(f"  ⚠️  {len(dupes)} duplicate barcodes! Examples: {dupes[:5]}")

    mat = sp_sparse.csr_matrix(
        (np.array(all_vals, dtype=np.uint32),
         (np.array(all_rows, dtype=np.int64),
          np.array(all_cols, dtype=np.int64))),
        shape=(n_regions, n_barcodes),
    )

    meta_df = pd.DataFrame(meta_records)
    if len(meta_df) > 0:
        meta_df = meta_df.set_index("barcode")

    print(f"\n  ✅ Final matrix: {n_regions:,} peaks × {n_barcodes:,} barcodes, "
          f"{mat.nnz:,} nonzero entries")

    return mat, all_barcodes, region_ids, meta_df


print("✅  build_fragment_matrix() defined")

### 3a — Run matrix construction for all species

This will take a while for the first run.  Results are saved to disk
(feather) so subsequent runs can just load from file.

In [ ]:
# === Build or load single-cell matrices for all species ===

SPECIES_MATRICES = {}  # species → (csr_matrix, barcodes, region_ids, meta_df)
FORCE_REBUILD = False  # Set True to re-compute even if saved files exist

for species in SPECIES_LIST:
    print(f"\n{'='*60}")
    print(f"  {species}")
    print(f"{'='*60}")

    sp_dir = os.path.join(OUTPUT_DIR, species)
    os.makedirs(sp_dir, exist_ok=True)
    mat_file = os.path.join(sp_dir, "sc_matrix.npz")
    bc_file  = os.path.join(sp_dir, "barcodes.parquet")
    reg_file = os.path.join(sp_dir, "region_ids.parquet")
    meta_file = os.path.join(sp_dir, "barcode_metadata.parquet")

    if species not in SPECIES_PEAK_FILES:
        print(f"  ⚠️  No peak file for {species}, skipping")
        continue

    if (os.path.exists(mat_file) and os.path.exists(meta_file)
            and not FORCE_REBUILD):
        print(f"  Loading from cache: {sp_dir}")
        mat = sp_sparse.load_npz(mat_file)
        barcodes = pd.read_parquet(bc_file)["barcode"].tolist()
        region_ids = pd.read_parquet(reg_file)["region_id"].tolist()
        meta_df = pd.read_parquet(meta_file)
        SPECIES_MATRICES[species] = (mat, barcodes, region_ids, meta_df)
        print(f"  Matrix: {mat.shape[0]:,} regions × {mat.shape[1]:,} barcodes")
        continue

    # Build from scratch
    mat, barcodes, region_ids, meta_df = build_fragment_matrix(
        species=species,
        cell_data=ALL_CELL_DATA[species],
        fragments_dict=ALL_FRAG_DICTS[species],
        peak_file=SPECIES_PEAK_FILES[species],
    )

    if mat is None:
        print(f"  ⚠️  Failed to build matrix for {species}")
        continue

    SPECIES_MATRICES[species] = (mat, barcodes, region_ids, meta_df)
    print(f"\n  ✅ Matrix: {mat.shape[0]:,} regions × {mat.shape[1]:,} barcodes")
    print(f"     Non-zero: {mat.nnz:,}  ({mat.nnz / np.prod(mat.shape) * 100:.3f}%)")

    # Save
    sp_sparse.save_npz(mat_file, mat)
    pd.DataFrame({"barcode": barcodes}).to_parquet(bc_file)
    pd.DataFrame({"region_id": region_ids}).to_parquet(reg_file)
    meta_df.to_parquet(meta_file)
    print(f"  💾 Saved to {sp_dir}")

## 4 — Pseudobulk aggregation

Sum single-cell counts into pseudobulk profiles, grouped by:
- **cell_type** (always)
- **donor** (Individual)
- **run** (Sequencing.Run, where available)

This creates a peaks × pseudobulk-samples DataFrame suitable for DESeq2.

In [ ]:
def create_pseudobulk(
    mat,
    barcodes: list,
    region_ids: list,
    meta_df: pd.DataFrame,
    group_by: list[str] = ["cell_type", "donor"],
    min_cells: int = 5,
) -> pd.DataFrame:
    """
    Sum single-cell counts into pseudobulk profiles.

    Parameters
    ----------
    mat : csr_matrix
        Peaks × barcodes sparse matrix.
    barcodes : list
        Barcode names (columns of mat).
    region_ids : list
        Region IDs (rows of mat).
    meta_df : pd.DataFrame
        Barcode metadata with columns matching `group_by`.
    group_by : list of str
        Columns in meta_df to group barcodes.
    min_cells : int
        Minimum barcodes per group to create a pseudobulk.

    Returns
    -------
    pd.DataFrame
        Peaks × pseudobulk samples, with MultiIndex columns.
    """
    # Ensure all group columns exist
    for col in group_by:
        if col not in meta_df.columns:
            raise ValueError(f"Column '{col}' not in metadata: {list(meta_df.columns)}")

    # Assign column index to each barcode
    bc_to_idx = {bc: i for i, bc in enumerate(barcodes)}

    # Group
    groups = meta_df.groupby(group_by, observed=True)

    pseudobulk = {}
    group_info = []

    for group_key, group_df in groups:
        if len(group_df) < min_cells:
            continue

        # Get column indices for this group
        col_indices = [bc_to_idx[bc] for bc in group_df.index if bc in bc_to_idx]
        if len(col_indices) < min_cells:
            continue

        # Sum columns
        sub = mat[:, col_indices]
        counts = np.asarray(sub.sum(axis=1)).ravel()

        # Label
        if isinstance(group_key, tuple):
            label = "__".join(str(x) for x in group_key)
        else:
            label = str(group_key)

        # Clean label for filesystem safety
        label_clean = re.sub(r"[/\\:*?\"<>|]", "_", label)
        label_clean = re.sub(r"\s+", "_", label_clean)

        pseudobulk[label_clean] = counts
        rec = {col: val for col, val in zip(group_by,
               group_key if isinstance(group_key, tuple) else [group_key])}
        rec["label"] = label_clean
        rec["n_cells"] = len(col_indices)
        group_info.append(rec)

    df = pd.DataFrame(pseudobulk, index=region_ids)
    info_df = pd.DataFrame(group_info)

    return df, info_df


print("✅  create_pseudobulk() defined")

### 4a — Run pseudobulk for all species

In [ ]:
# === Create pseudobulk matrices for all species ===

# Grouping: cell_type × donor
# Change this list to add "run" for cell_type × donor × run
PSEUDOBULK_GROUPBY = ["cell_type", "donor"]
MIN_CELLS_PER_GROUP = 5

PSEUDOBULK_MATRICES = {}

for species in SPECIES_LIST:
    if species not in SPECIES_MATRICES:
        print(f"Skipping {species}: no matrix")
        continue

    print(f"\n{'='*60}")
    print(f"  {species} — Pseudobulk by {PSEUDOBULK_GROUPBY}")
    print(f"{'='*60}")

    mat, barcodes, region_ids, meta_df = SPECIES_MATRICES[species]

    # For Human, 'Sequencing.Run' may not exist — adapt groupby
    actual_groupby = [g for g in PSEUDOBULK_GROUPBY if g in meta_df.columns]
    if not actual_groupby:
        print(f"  ⚠️  No matching group columns in metadata, skipping")
        continue

    pb_df, info_df = create_pseudobulk(
        mat, barcodes, region_ids, meta_df,
        group_by=actual_groupby,
        min_cells=MIN_CELLS_PER_GROUP,
    )

    PSEUDOBULK_MATRICES[species] = (pb_df, info_df)

    print(f"  Matrix: {pb_df.shape[0]:,} peaks × {pb_df.shape[1]} pseudobulk samples")
    print(f"  Groups with ≥{MIN_CELLS_PER_GROUP} cells: {len(info_df)}")
    print(f"\n  Group summary:")
    print(info_df.to_string(index=False))

## 5 — Save pseudobulk matrices

In [ ]:
# === Save pseudobulk count matrices and group info ===

for species in SPECIES_LIST:
    if species not in PSEUDOBULK_MATRICES:
        continue

    pb_df, info_df = PSEUDOBULK_MATRICES[species]
    sp_dir = os.path.join(OUTPUT_DIR, species)
    os.makedirs(sp_dir, exist_ok=True)

    # Save count matrix
    out_counts = os.path.join(sp_dir, "pseudobulk_counts.parquet")
    pb_df.index.name = "region_id"
    pb_df.reset_index().to_parquet(out_counts)

    # Also save as TSV for easy inspection / DESeq2 import
    out_tsv = os.path.join(sp_dir, "pseudobulk_counts.tsv")
    pb_df.to_csv(out_tsv, sep="\t")

    # Save group info
    out_info = os.path.join(sp_dir, "pseudobulk_groups.parquet")
    info_df.to_parquet(out_info)
    out_info_tsv = os.path.join(sp_dir, "pseudobulk_groups.tsv")
    info_df.to_csv(out_info_tsv, sep="\t", index=False)

    print(f"✅  {species:<15} → {sp_dir}")
    print(f"    counts:  {out_counts}  ({os.path.getsize(out_counts)/1e6:.1f} MB)")
    print(f"    info:    {out_info_tsv}")

## 6 — Quick QC: inspect results

In [ ]:
# === Quick inspection of one species ===

INSPECT_SPECIES = "Human"

if INSPECT_SPECIES in PSEUDOBULK_MATRICES:
    pb_df, info_df = PSEUDOBULK_MATRICES[INSPECT_SPECIES]

    print(f"=== {INSPECT_SPECIES} Pseudobulk ===")
    print(f"Shape: {pb_df.shape}")
    print(f"\nColumn sums (total fragments per pseudobulk):")
    col_sums = pb_df.sum(axis=0).sort_values(ascending=False)
    for col, val in col_sums.head(15).items():
        print(f"  {col:<50} {val:>12,.0f}")

    print(f"\nRow sums (fragments per peak, top 10):")
    row_sums = pb_df.sum(axis=1).sort_values(ascending=False)
    for idx, val in row_sums.head(10).items():
        print(f"  {idx:<40} {val:>12,.0f}")

    print(f"\nZero-count peaks: {(pb_df.sum(axis=1) == 0).sum():,} / {pb_df.shape[0]:,}")
    print(f"\nGroup info:")
    print(info_df)
else:
    print(f"{INSPECT_SPECIES} not available. Available: {list(PSEUDOBULK_MATRICES.keys())}")

In [ ]:
# === Summary of all saved files ===

print(f"\n{'='*70}")
print(f"  All saved files in {OUTPUT_DIR}")
print(f"{'='*70}\n")

for species in SPECIES_LIST:
    sp_dir = os.path.join(OUTPUT_DIR, species)
    if not os.path.isdir(sp_dir):
        continue
    print(f"📁 {species}/")
    for f in sorted(os.listdir(sp_dir)):
        fpath = os.path.join(sp_dir, f)
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"   {f:<45} {size_mb:>8.1f} MB")
    print()